# Exp 17: Jaccard Similarity Analysis

**Goal:** Compute per-problem set-level similarity (Jaccard) between all rater pairs across Human A, Human B, V1 Enriched, V2 Enriched, and V3.

Jaccard ignores shared zeros (both raters agree no gap), so it handles the sparse matrix correctly.

**Output:**
- Per-problem Jaccard and F1 for all 10 rater pairs
- Summary table per student and overall

## Formula

$$
J(A, B) = \frac{|A \cap B|}{|A \cup B|}
$$

Where:
- $A$ = set of KC gaps tagged by Rater A for a given problem
- $B$ = set of KC gaps tagged by Rater B for the same problem
- $|A \cap B|$ = number of KCs both raters tagged as gaps
- $|A \cup B|$ = number of KCs at least one rater tagged as a gap

## Range

- $J = 1.0$ means both raters tagged the exact same set of gaps
- $J = 0.0$ means no overlap at all
- $J$ is undefined when both sets are empty (both raters said no gaps), so we skip those problems

## Precision / Recall / F1

**Precision**

$$
Precision = \frac{|V3 \cap Human|}{|V3|}
$$

Out of everything V3 tagged, how many were correct?

- Numerator: KCs that both V3 and Human flagged
- Denominator: total KCs that V3 flagged

**Recall**

$$
Recall = \frac{|V3 \cap Human|}{|Human|}
$$

Out of everything the Human tagged, how many did V3 catch?

- Numerator: same intersection
- Denominator: total KCs that the Human flagged

**F1**

$$
F1 = \frac{2 \times Precision \times Recall}{Precision + Recall} = \frac{2 \times |V3 \cap Human|}{|V3| + |Human|}
$$

Which is exactly Dice.

Implementation note: the metrics in this notebook are computed with scikit-learn on binary KC vectors, rather than by hand.

## Full Example

Human A: {If/Else, LogicAndNotOr, LogicCompareNum} -> $|Human| = 3$

V3: {LogicAndNotOr, For} -> $|V3| = 2$

Intersection: {LogicAndNotOr} -> $1$

Precision = $1/2 = 0.50$ -> "V3 flagged 2 things, 1 was right"

Recall = $1/3 = 0.33$ -> "Human found 3 things, V3 caught 1"

F1 = $2 \times 1 / (2 + 3) = 2/5 = 0.40$

Now compare to Jaccard on the same example:

Union = {If/Else, LogicAndNotOr, LogicCompareNum, For} -> $4$

Jaccard = $1/4 = 0.25$

F1 = 0.40, Jaccard = 0.25. Same data, Jaccard is harsher because the union includes both what V3 added wrong (For) and what V3 missed (If/Else, LogicCompareNum). F1 splits those into two separate penalties (precision and recall) and averages them more gently.

## Why Jaccard for This Task

Simple percentage agreement (matching cells in the matrix) would be inflated by sparsity. Most cells in the problem x KC matrix are 0 (no gap). Two raters could "agree" on 95% of cells just because they both left most cells empty. Jaccard ignores shared zeros entirely and only measures agreement on cells where at least one rater said "this is a gap."


In [37]:
import json
from pathlib import Path
from itertools import combinations

import pandas as pd
from IPython.display import display, Markdown
from sklearn.metrics import jaccard_score, precision_score, recall_score, f1_score

ROOT = Path.cwd()
if not (ROOT / 'results').exists() and (ROOT.parent / 'results').exists():
    ROOT = ROOT.parent
if not (ROOT / 'results').exists() and (ROOT.parent.parent / 'results').exists():
    ROOT = ROOT.parent.parent

print(f'Project root: {ROOT}')

Project root: /mnt/d/Projects/kintsugi


In [31]:
# === File paths ===

STUDENTS = ['10155', '14475', '14476']

RATER_FILES = {
    'Human A': {
        '10155': ROOT / 'dataset/Rater_KC_Tags/kc_annotations_Pranay Ghuge_10155_1774736175604.json',
        '14475': ROOT / 'dataset/Rater_KC_Tags/kc_annotations_Pranay Ghuge_14475_1775593592812.json',
        '14476': ROOT / 'dataset/Rater_KC_Tags/kc_annotations_Pranay Ghuge_14476_1775593175294.json',
    },
    'Human B': {
        '10155': ROOT / 'dataset/Rater_KC_Tags/kc_annotations_Arundhati Das_10155_1774820622134.json',
        '14475': ROOT / 'dataset/Rater_KC_Tags/kc_annotations_Arundhati Das_14475_1775593132134.json',
        '14476': ROOT / 'dataset/Rater_KC_Tags/kc_annotations_Arundhati Das_14476_1775691915454.json',
    },
    'V1 Enriched': {
        '10155': ROOT / 'results/human_validation/llm_annotations_10155.json',
        '14475': ROOT / 'results/human_validation/llm_annotations_14475.json',
        '14476': ROOT / 'results/human_validation/llm_annotations_14476.json',
    },
    'V2 Enriched': {
        '10155': ROOT / 'results/human_validation/llm_enriched_annotations_v2_10155.json',
        '14475': ROOT / 'results/human_validation/llm_enriched_annotations_v2_14475.json',
        '14476': ROOT / 'results/human_validation/llm_enriched_annotations_v2_14476.json',
    },
    'V3 (CCPP)': {
        '10155': ROOT / 'results/human_validation/llm_v3_annotations_10155.json',
        '14475': ROOT / 'results/human_validation/llm_v3_annotations_14475.json',
        '14476': ROOT / 'results/human_validation/llm_v3_annotations_14476.json',
    },
}

KC_COLUMNS = [
    'If/Else', 'NestedIf', 'While', 'For', 'NestedFor',
    'Math+-*/', 'Math%', 'LogicAndNotOr', 'LogicCompareNum', 'LogicBoolean',
    'StringFormat', 'StringConcat', 'StringIndex', 'StringLen',
    'StringEqual', 'CharEqual', 'ArrayIndex', 'DefFunction'
]

In [32]:
# === Load annotations ===
# Returns dict: {composite_pid: set_of_gap_kcs}

def load_human_annotations(filepath):
    """Load human rater JSON (format: {annotations: {pid: {gaps: [...]}}})."""
    with open(filepath, 'r', encoding='utf-8') as f:
        data = json.load(f)
    student_id = str(data.get('student_id', data.get('studentId', 'unknown')))
    annotations = {}
    for pid, item in data.get('annotations', {}).items():
        gaps = item.get('gaps', []) if isinstance(item, dict) else (item if isinstance(item, list) else [])
        annotations[f'{student_id}_{pid}'] = {kc for kc in gaps if kc in KC_COLUMNS}
    return annotations


def load_llm_annotations(filepath):
    """Load LLM annotation JSON. Handles both V1/V2 and V3 formats."""
    with open(filepath, 'r', encoding='utf-8') as f:
        data = json.load(f)
    
    student_id = str(data.get('student_id', data.get('studentId', 'unknown')))
    annotations = {}
    
    # Try V3 format first (has raw_responses with parsed_response.knowledge_gaps)
    raw_responses = data.get('raw_responses', {})
    if raw_responses:
        for pid, item in raw_responses.items():
            parsed = item.get('parsed_response', {})
            gaps = parsed.get('knowledge_gaps', [])
            # Convert gap dicts to KC strings if needed
            kc_set = set()
            for g in gaps:
                if isinstance(g, str):
                    if g in KC_COLUMNS:
                        kc_set.add(g)
                elif isinstance(g, dict):
                    kc = g.get('kc_tag', g.get('missing_concept', ''))
                    if kc in KC_COLUMNS:
                        kc_set.add(kc)
            annotations[f'{student_id}_{pid}'] = kc_set
        return annotations
    
    # V1/V2 format (has annotations with gaps)
    for pid, item in data.get('annotations', {}).items():
        gaps = item.get('gaps', []) if isinstance(item, dict) else (item if isinstance(item, list) else [])
        kc_set = set()
        for g in gaps:
            if isinstance(g, str) and g in KC_COLUMNS:
                kc_set.add(g)
            elif isinstance(g, dict):
                kc = g.get('kc_tag', g.get('missing_concept', ''))
                if kc in KC_COLUMNS:
                    kc_set.add(kc)
        annotations[f'{student_id}_{pid}'] = kc_set
    return annotations


# Load all raters
all_annotations = {}  # {rater_name: {composite_pid: set_of_kcs}}

for rater_name, student_files in RATER_FILES.items():
    merged = {}
    for sid, fp in student_files.items():
        if not fp.exists():
            print(f'WARNING: {fp} not found, skipping')
            continue
        if rater_name.startswith('Human'):
            ann = load_human_annotations(fp)
        else:
            ann = load_llm_annotations(fp)
        merged.update(ann)
    all_annotations[rater_name] = merged
    print(f'{rater_name}: {len(merged)} problems loaded')

# Get common problem IDs across all raters
common_pids = set.intersection(*[set(ann.keys()) for ann in all_annotations.values()])
print(f'\nCommon problems across all raters: {len(common_pids)}')

Human A: 150 problems loaded
Human B: 150 problems loaded
V1 Enriched: 146 problems loaded
V2 Enriched: 146 problems loaded
V3 (CCPP): 146 problems loaded

Common problems across all raters: 146


In [40]:
# === Compute Jaccard similarity ===

def annotation_vector(gap_set):
    """Convert a KC gap set into a binary vector over KC_COLUMNS."""
    return [1 if kc in gap_set else 0 for kc in KC_COLUMNS]


def overlap_metrics(set_a, set_b):
    """Return precision, recall, F1, and Jaccard using scikit-learn."""
    if not set_a and not set_b:
        return None, None, None, None

    y_true = annotation_vector(set_a)
    y_pred = annotation_vector(set_b)

    precision = precision_score(y_true, y_pred, zero_division=0)
    recall = recall_score(y_true, y_pred, zero_division=0)
    f1_value = f1_score(y_true, y_pred, zero_division=0)
    jaccard_value = jaccard_score(y_true, y_pred, zero_division=0)

    return precision, recall, f1_value, jaccard_value


rater_names = list(all_annotations.keys())
rater_pairs = list(combinations(rater_names, 2))

# Per-problem metrics for each pair
records = []
for pid in sorted(common_pids, key=lambda x: (x.split('_')[0], int(x.split('_')[1]))):
    student_id = pid.split('_')[0]
    problem_id = pid.split('_')[1]
    row = {'student_id': student_id, 'problem_id': problem_id, 'composite_pid': pid}
    for r1, r2 in rater_pairs:
        precision, recall, f1_value, jaccard_value = overlap_metrics(
            all_annotations[r1].get(pid, set()),
            all_annotations[r2].get(pid, set())
        )
        pair_label = f'{r1} vs {r2}'
        row[f'{pair_label} | Precision'] = precision
        row[f'{pair_label} | Recall'] = recall
        row[f'{pair_label} | F1'] = f1_value
        row[f'{pair_label} | Jaccard'] = jaccard_value
    records.append(row)

df_metrics = pd.DataFrame(records)
print(f'Total problems: {len(df_metrics)}')


Total problems: 146


In [43]:
# === Summary statistics ===

metric_cols = [c for c in df_metrics.columns if c.endswith(' | Jaccard')]

def build_summary_table(frame):
    rows = []
    for metric_col in metric_cols:
        pair = metric_col.rsplit(' | ', 1)[0]
        precision_col = f'{pair} | Precision'
        recall_col = f'{pair} | Recall'
        f1_col = f'{pair} | F1'
        rows.append({
            'Pair': pair,
            'Precision': frame[precision_col].mean(),
            'Recall': frame[recall_col].mean(),
            'F1': frame[f1_col].mean(),
            'Jaccard': frame[metric_col].mean(),
            'n': int(frame[metric_col].notna().sum()),
        })
    table = pd.DataFrame(rows).sort_values('Jaccard', ascending=False).reset_index(drop=True)
    for column in ['Precision', 'Recall', 'F1', 'Jaccard']:
        table[column] = table[column].map(lambda x: f'{x:0.3f}')
    return table

def build_final_rater_table(frame):
    rater_order = ['Human A', 'Human B', 'V1 Enriched', 'V2 Enriched', 'V3 (CCPP)']
    rows = []
    for metric_name in ['Precision', 'Recall', 'F1', 'Jaccard']:
        row = {'Metric': metric_name}
        for rater in rater_order:
            if metric_name == 'Jaccard':
                values = []
                for other in rater_order:
                    if other == rater:
                        continue
                    col = f'{rater} vs {other} | Jaccard'
                    reverse_col = f'{other} vs {rater} | Jaccard'
                    if col in frame.columns:
                        values.append(frame[col].mean())
                    elif reverse_col in frame.columns:
                        values.append(frame[reverse_col].mean())
                row[rater] = sum(values) / len(values) if values else None
            else:
                values = []
                for other in rater_order:
                    if other == rater:
                        continue
                    col = f'{rater} vs {other} | {metric_name}'
                    reverse_col = f'{other} vs {rater} | {metric_name}'
                    if col in frame.columns:
                        values.append(frame[col].mean())
                    elif reverse_col in frame.columns:
                        values.append(frame[reverse_col].mean())
                row[rater] = sum(values) / len(values) if values else None
        rows.append(row)
    table = pd.DataFrame(rows)
    for rater in rater_order:
        table[rater] = table[rater].map(lambda x: f'{x:0.3f}' if pd.notna(x) else '')
    return table

for sid in STUDENTS:
    subset = df_metrics[df_metrics['student_id'] == sid]
    display(Markdown(f'## Student {sid} Summary'))
    display(Markdown(f'Problems: {len(subset)}'))
    display(build_summary_table(subset))

display(Markdown('## Final Combined Score Across All Students'))
display(Markdown('This is the weighted overall average across all problems from all 3 students.'))
display(build_final_rater_table(df_metrics))


## Student 10155 Summary

Problems: 46

,Pair,Precision,Recall,F1,Jaccard,n
0,Human A vs Human B,0.727,0.661,0.663,0.543,15
1,V1 Enriched vs V2 Enriched,0.539,0.708,0.598,0.518,20
2,Human A vs V3 (CCPP),0.764,0.587,0.621,0.512,14
3,Human B vs V3 (CCPP),0.691,0.574,0.597,0.489,15
4,V1 Enriched vs V3 (CCPP),0.573,0.554,0.534,0.414,17
5,V2 Enriched vs V3 (CCPP),0.613,0.481,0.507,0.394,18
6,Human A vs V2 Enriched,0.481,0.431,0.441,0.335,18
7,Human A vs V1 Enriched,0.559,0.393,0.450,0.328,17
8,Human B vs V1 Enriched,0.500,0.378,0.423,0.312,18
9,Human B vs V2 Enriched,0.390,0.399,0.383,0.280,19


## Student 14475 Summary

Problems: 50

...

Problems: 50

,Pair,Precision,Recall,F1,Jaccard,n
0,V1 Enriched vs V2 Enriched,0.583,0.717,0.601,0.497,27
1,V2 Enriched vs V3 (CCPP),0.781,0.545,0.591,0.446,25
2,Human A vs Human B,0.694,0.465,0.538,0.398,24
3,Human A vs V3 (CCPP),0.738,0.443,0.521,0.392,24
4,Human A vs V2 Enriched,0.573,0.460,0.471,0.349,25
5,V1 Enriched vs V3 (CCPP),0.557,0.454,0.444,0.325,26
6,Human A vs V1 Enriched,0.501,0.332,0.388,0.280,26
7,Human B vs V3 (CCPP),0.481,0.355,0.387,0.275,24
8,Human B vs V2 Enriched,0.374,0.417,0.374,0.273,25
9,Human B vs V1 Enriched,0.302,0.266,0.270,0.197,26


## Final Combined Score Across All Students

This is the weighted overall average across all problems from all 3 students.

,Metric,Human A,Human B,V1 Enriched,V2 Enriched,V3 (CCPP)
0,Precision,0.575,0.477,0.488,0.536,0.616
1,Recall,0.426,0.394,0.449,0.507,0.461
2,F1,0.464,0.412,0.437,0.485,0.491
3,Jaccard,0.354,0.310,0.338,0.376,0.375
